In [1]:
import pandas as pd

# Load data set
rev_df = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\AMDARI\Montra Hospitality Project\Data\Initial Merged Data\merged_data.csv")

In [2]:
rev_df.columns

Index(['review_id', 'booking_id', 'guest_id', 'hotel_id', 'review_text',
       'rating_score', 'review_date', 'sentiment_label', 'room_type',
       'booking_channel', 'check_in_date', 'check_out_date', 'total_amount',
       'stay_duration', 'customer_type', 'loyalty_status', 'country',
       'previous_stays', 'hotel_name', 'location', 'star_rating',
       'total_rooms'],
      dtype='object')

In [3]:
# Standardizing Column names

rev_df.columns = (
    rev_df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

In [4]:
rev_df.columns

Index(['review_id', 'booking_id', 'guest_id', 'hotel_id', 'review_text',
       'rating_score', 'review_date', 'sentiment_label', 'room_type',
       'booking_channel', 'check_in_date', 'check_out_date', 'total_amount',
       'stay_duration', 'customer_type', 'loyalty_status', 'country',
       'previous_stays', 'hotel_name', 'location', 'star_rating',
       'total_rooms'],
      dtype='object')

In [5]:
# Ensuring no missing values

rev_df.isnull().sum()

review_id          0
booking_id         0
guest_id           0
hotel_id           0
review_text        0
rating_score       0
review_date        0
sentiment_label    0
room_type          0
booking_channel    0
check_in_date      0
check_out_date     0
total_amount       0
stay_duration      0
customer_type      0
loyalty_status     0
country            0
previous_stays     0
hotel_name         0
location           0
star_rating        0
total_rooms        0
dtype: int64

In [6]:
# Ensuring no duplicates

rev_df.duplicated().sum()

np.int64(0)

In [7]:
!pip install contractions

In [8]:
# Cleaning review text

import re
import contractions

def clean_review(text):

    text = str(text)

    # Expand contractions
    text = contractions.fix(text)

    # Lowercase
    text = text.lower()

    # Remove HTML
    text = re.sub(r'<.*?>', ' ', text)

    # Remove filler phrases
    filler_phrases = [
        'nothing more and nothing less',
        'about what we expected',
        'might consider it again',
        'would neither recommend nor avoid it'
    ]

    for phrase in filler_phrases:
        text = text.replace(phrase, '')

    # Remove punctuation and digits
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
rev_df["clean_review"] = rev_df["review_text"].apply(clean_review)

In [10]:
rev_df[["review_text", "clean_review"]].head()

,review_text,clean_review
0,The overall price was fair for what we paid. M...,the overall price was fair for what we paid mi...
1,The room was nothing out of the ordinary for w...,the room was nothing out of the ordinary for w...
2,"Unfortunately, the pool was outdated — it clea...",unfortunately the pool was outdated it clearly...
3,We were disappointed that the neighborhood was...,we were disappointed that the neighborhood was...
4,The room service was standard hotel fare for w...,the room service was standard hotel fare for w...


In [11]:
!pip install spacy

In [12]:
!pip cache purge
!python -m pip install --upgrade pip wheel setuptools
!python -m spacy download en_core_web_sm

Files removed: 8 (13.1 MB)
Directories removed: 21
   ---------------------------------------- 0.0/875.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/875.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/875.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/875.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/875.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/875.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/875.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/875.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/875.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/875.8 kB ? eta -:--:--
   ---------------------- --------------- 524.3/875.8 kB 297.5 kB/s eta 0:00:02
   ---------------------- --------------- 524.3/875.8 kB 297.5 kB/s eta 0:00:02
   ---------------------- --------------- 524.3/875.8 kB 297.5 kB/s eta

In [13]:
# Lemmatization

import spacy
nlp = spacy.load("en_core_web_sm")



In [14]:
# Lemmatization

def lemmatize_text(text):

    doc = nlp(text)

    return " ".join(
        token.lemma_
        for token in doc
    )

rev_df["clean_review"] = (
    rev_df["clean_review"]
    .apply(lemmatize_text)
)

##### Standardize Variables

In [15]:
# Sentiment Labels

rev_df["sentiment_label"] = (
    rev_df["sentiment_label"]
      .str.lower()
      .str.strip()
)

sentiment_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

rev_df["sentiment_encoded"] = (
    rev_df["sentiment_label"]
      .map(sentiment_map)
)

In [16]:
rev_df.head(1)

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms,clean_review,sentiment_encoded
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,Leisure,Unknown,France,4,The Aberdeen Grand,Aberdeen,4,118,the overall price be fair for what we pay migh...,0


In [17]:
# Dates: converting dates to pd-datetime

date_cols = [
    "review_date",
    "check_in_date",
    "check_out_date"
]

for col in date_cols:
    rev_df[col] = pd.to_datetime(
        rev_df[col],
        errors="coerce"
    )

In [18]:
# Numeric columns: converting numeric features to numeric data type

numeric_cols = [
    "rating_score",
    "total_amount",
    "stay_duration",
    "previous_stays"
]

for col in numeric_cols:
    rev_df[col] = pd.to_numeric(
        rev_df[col],
        errors="coerce"
    )

In [19]:
rev_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   review_id          35000 non-null  int64         
 1   booking_id         35000 non-null  int64         
 2   guest_id           35000 non-null  int64         
 3   hotel_id           35000 non-null  int64         
 4   review_text        35000 non-null  object        
 5   rating_score       35000 non-null  float64       
 6   review_date        35000 non-null  datetime64[ns]
 7   sentiment_label    35000 non-null  object        
 8   room_type          35000 non-null  object        
 9   booking_channel    35000 non-null  object        
 10  check_in_date      35000 non-null  datetime64[ns]
 11  check_out_date     35000 non-null  datetime64[ns]
 12  total_amount       35000 non-null  float64       
 13  stay_duration      35000 non-null  int64         
 14  custom

###### Feature Engineering for NLP and Machine Learning

In [20]:
# Review Length

rev_df["word_count"] = (
    rev_df["clean_review"]
      .str.split()
      .str.len()
)

In [21]:
# Character Length

rev_df["char_count"] = (
    rev_df["clean_review"]
      .str.len()
)

In [22]:
# Stay duration validation

rev_df["stay_duration_calc"] = (
    rev_df["check_out_date"]
    - rev_df["check_in_date"]
).dt.days

In [23]:
# Price per night

rev_df["price_per_night"] = (
    rev_df["total_amount"]
    / rev_df["stay_duration"]
).round(2)

##### Preparing NLP Corpus

In [24]:
corpus = rev_df["clean_review"].tolist()

In [25]:
corpus[:5]

['the overall price be fair for what we pay might not',
 'the room be nothing out of the ordinary for what we pay the overall price exceed our expectation a great deal and the quality easily match the price',
 'unfortunately the pool be outdate it clearly need some upkeep would not recommend it to other',
 'we be disappoint that the neighborhood be disappointingly out of the way meanwhile sadly the room turn out to be grimy',
 'the room service be standard hotel fare for what we pay that say the bedding be in decent shape for what we pay on top of that the cost of the stay be reasonable enough for what we pay']

In [26]:
rev_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   review_id           35000 non-null  int64         
 1   booking_id          35000 non-null  int64         
 2   guest_id            35000 non-null  int64         
 3   hotel_id            35000 non-null  int64         
 4   review_text         35000 non-null  object        
 5   rating_score        35000 non-null  float64       
 6   review_date         35000 non-null  datetime64[ns]
 7   sentiment_label     35000 non-null  object        
 8   room_type           35000 non-null  object        
 9   booking_channel     35000 non-null  object        
 10  check_in_date       35000 non-null  datetime64[ns]
 11  check_out_date      35000 non-null  datetime64[ns]
 12  total_amount        35000 non-null  float64       
 13  stay_duration       35000 non-null  int64     

In [27]:
clean_df= rev_df

In [28]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   review_id           35000 non-null  int64         
 1   booking_id          35000 non-null  int64         
 2   guest_id            35000 non-null  int64         
 3   hotel_id            35000 non-null  int64         
 4   review_text         35000 non-null  object        
 5   rating_score        35000 non-null  float64       
 6   review_date         35000 non-null  datetime64[ns]
 7   sentiment_label     35000 non-null  object        
 8   room_type           35000 non-null  object        
 9   booking_channel     35000 non-null  object        
 10  check_in_date       35000 non-null  datetime64[ns]
 11  check_out_date      35000 non-null  datetime64[ns]
 12  total_amount        35000 non-null  float64       
 13  stay_duration       35000 non-null  int64     

In [29]:
import os

# Define the folder path
folder_path = r"c:\Users\HP\OneDrive\Documents\AMDARI\Montra Hospitality Project\Notebook"

# Create the full save path for your new CSV file
save_path = os.path.join(folder_path, "cleaned_processed_data.csv")

# Save the dataframe to the folder (index=False stops it from adding an extra column of numbers)
clean_df.to_csv(save_path, index=False)

print(f"File successfully saved to: {save_path}")

File successfully saved to: c:\Users\HP\OneDrive\Documents\AMDARI\Montra Hospitality Project\Notebook\cleaned_processed_data.csv


##### Create Quality Report Container

In [30]:
quality_report = {
    "Check": [],
    "Issue_Detected": [],
    "Affected_Rows": [],
    "Action_Taken": []
}

In [31]:
# Missing Values Check

missing_values = clean_df.isnull().sum()

for column, count in missing_values.items():

    if count > 0:

        quality_report["Check"].append("Missing Values")

        quality_report["Issue_Detected"].append(
            f"{column} contains {count} missing values"
        )

        quality_report["Affected_Rows"].append(count)

        quality_report["Action_Taken"].append(
            "Imputed or flagged for review"
        )

In [32]:
# Duplicate records check

duplicate_count = clean_df.duplicated().sum()

quality_report["Check"].append(
    "Duplicate Records"
)

quality_report["Issue_Detected"].append(
    f"{duplicate_count} duplicate rows found"
)

quality_report["Affected_Rows"].append(
    duplicate_count
)

quality_report["Action_Taken"].append(
    "Removed duplicates"
)

In [33]:
# Clean_review text quality check

short_reviews = clean_df[
    clean_df["clean_review"].str.split().str.len() < 3
]

quality_report["Check"].append(
    "Review Length"
)

quality_report["Issue_Detected"].append(
    f"{len(short_reviews)} reviews too short"
)

quality_report["Affected_Rows"].append(
    len(short_reviews)
)

quality_report["Action_Taken"].append(
    "Flagged for manual review"
)

In [34]:
# Sentiment and Rating consistency check

def expected_sentiment(score):

    if score >= 4:
        return "positive"

    elif score <= 2:
        return "negative"

    else:
        return "neutral"

# Apply rule
clean_df["expected_sentiment"] = (
    clean_df["rating_score"]
    .apply(expected_sentiment)
)

# Find inconsistencies
mismatches = clean_df[
    clean_df["expected_sentiment"] != clean_df["sentiment_label"]
]

# Log result
quality_report["Check"].append(
    "Sentiment Validation"
)

quality_report["Issue_Detected"].append(
    f"{len(mismatches)} sentiment mismatches"
)

quality_report["Affected_Rows"].append(
    len(mismatches)
)

quality_report["Action_Taken"].append(
    "Flagged for verification"
)

In [35]:
# Rating Check

invalid_ratings = clean_df[
    (clean_df["rating_score"] < 1)
    |
    (clean_df["rating_score"] > 5)
]

# Log result
quality_report["Check"].append(
    "Rating Range"
)

quality_report["Issue_Detected"].append(
    f"{len(invalid_ratings)} invalid ratings"
)

quality_report["Affected_Rows"].append(
    len(invalid_ratings)
)

quality_report["Action_Taken"].append(
    "Flagged for correction"
)

In [36]:
# Review date validation

today = pd.Timestamp.today()

future_dates = clean_df[
    clean_df["review_date"] > today
]

# Log result
quality_report["Check"].append(
    "Date Validation"
)

quality_report["Issue_Detected"].append(
    f"{len(future_dates)} future review dates"
)

quality_report["Affected_Rows"].append(
    len(future_dates)
)

quality_report["Action_Taken"].append(
    "Flagged for review"
)

In [37]:
# Checking price outliers

Q1 = clean_df["total_amount"].quantile(0.25)

Q3 = clean_df["total_amount"].quantile(0.75)

IQR = Q3 - Q1

outliers = clean_df[
    (clean_df["total_amount"] < Q1 - 1.5 * IQR)
    |
    (clean_df["total_amount"] > Q3 + 1.5 * IQR)
]

# Log results
quality_report["Check"].append(
    "Price Outliers"
)

quality_report["Issue_Detected"].append(
    f"{len(outliers)} outliers detected"
)

quality_report["Affected_Rows"].append(
    len(outliers)
)

quality_report["Action_Taken"].append(
    "Flagged for investigation"
)

In [38]:
# Checking for noise

noise_patterns = [
    "nothing more and nothing less",
    "about what we expected",
    "might consider it again"
]

noise_count = 0

for phrase in noise_patterns:

    noise_count += clean_df["clean_review"] \
                    .str.contains(
                        phrase,
                        case=False,
                        na=False
                    ).sum()

# Log results
quality_report["Check"].append(
    "Text Noise"
)

quality_report["Issue_Detected"].append(
    f"{noise_count} noisy reviews found"
)

quality_report["Affected_Rows"].append(
    noise_count
)

quality_report["Action_Taken"].append(
    "Removed during preprocessing"
)

In [39]:
# Data set integrity check (Critical Columns)

required_columns = [
    "clean_review",
    "sentiment_label",
    "rating_score"
]

missing_columns = [
    col
    for col in required_columns
    if col not in clean_df.columns
]

# Log result
quality_report["Check"].append(
    "Schema Validation"
)

quality_report["Issue_Detected"].append(
    str(missing_columns)
)

quality_report["Affected_Rows"].append(
    0
)

quality_report["Action_Taken"].append(
    "Verified dataset structure"
)

##### Create Quality Audit Report

In [40]:
quality_df = pd.DataFrame(
    quality_report
)

# Save Report
quality_df.to_csv(
    "data_quality_report.csv",
    index=False
)

##### Generate Summary Log

In [41]:
from datetime import datetime

with open(
    "quality_audit_log.txt",
    "w",
    encoding="utf-8"
) as log:

    log.write(
        "DATA QUALITY AUDIT REPORT\n"
    )

    log.write(
        "=" * 50 + "\n"
    )

    log.write(
        f"Audit Date: {datetime.now()}\n\n"
    )

    for _, row in quality_df.iterrows():

        log.write(
            f"Check: {row['Check']}\n"
        )

        log.write(
            f"Issue: {row['Issue_Detected']}\n"
        )

        log.write(
            f"Affected Records: {row['Affected_Rows']}\n"
        )

        log.write(
            f"Action Taken: {row['Action_Taken']}\n\n"
        )

In [42]:
quality_df.head(20)

,Check,Issue_Detected,Affected_Rows,Action_Taken
0,Duplicate Records,0 duplicate rows found,0,Removed duplicates
1,Review Length,0 reviews too short,0,Flagged for manual review
2,Sentiment Validation,10839 sentiment mismatches,10839,Flagged for verification
3,Rating Range,0 invalid ratings,0,Flagged for correction
4,Date Validation,0 future review dates,0,Flagged for review
5,Price Outliers,1551 outliers detected,1551,Flagged for investigation
6,Text Noise,0 noisy reviews found,0,Removed during preprocessing
7,Schema Validation,[],0,Verified dataset structure


##### Applying Fixes

In [43]:
# Review Length

review_length = (
    clean_df["clean_review"]
      .str.split()
      .str.len()
)

short_reviews = (
    review_length < 3
)

clean_df = clean_df[
    review_length >= 3
]


In [44]:
clean_df = rev_df[rev_df["rating_score"].notna()].copy()

In [45]:
# Sentiment mismatches

# Create sentiment rule from ratings
def derive_sentiment(score):

    if score >= 4:
        return "positive"

    elif score <= 2:
        return "negative"

    else:
        return "neutral"

# Apply rule
clean_df["derived_sentiment"] = (
    clean_df["rating_score"]
    .apply(derive_sentiment)
)

# find mismatch
mismatch_mask = (
    clean_df["sentiment_label"]
    !=
    clean_df["derived_sentiment"]
)

mismatch_count = mismatch_mask.sum()

print(mismatch_count)

clean_df.loc[
    mismatch_mask,
    "sentiment_label"
] = clean_df.loc[
    mismatch_mask,
    "derived_sentiment"
]

10839


In [46]:
# Cap Amount Outliers

Q1 = clean_df["total_amount"].quantile(0.25)

Q3 = clean_df["total_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = (
    Q1 - 1.5 * IQR
)

upper_limit = (
    Q3 + 1.5 * IQR
)

# Outliers count
outlier_count = (
    (
        clean_df["total_amount"] > upper_limit
    )
    |
    (
       clean_df["total_amount"] < lower_limit
    )
).sum()

print(outlier_count)

clean_df["total_amount"] = (
    clean_df["total_amount"]
    .clip(
        lower=lower_limit,
        upper=upper_limit
    )
)

clean_df["total_amount"] = (
   clean_df["total_amount"]
    .clip(
        lower=lower_limit,
        upper=upper_limit
    )
)

1551


In [47]:
# Save Data
clean_df.to_csv(
    "quality_checked.csv",
    index=False
)